In [19]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [24]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 ICML 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 정보가 포함된 div 태그 찾기
    for paper in soup.find_all("div", class_="paper"):
        # 제목 찾기
        title_tag = paper.find("p", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tag = paper.find("span", class_="authors")
        authors_cleaned = authors_tag.text.strip() if authors_tag else "Unknown"

        # PDF 링크 & 코드 링크 찾기
        pdf_link = None
        code_url = None

        links = paper.find("p", class_="links")
        if links:
            for a in links.find_all("a"):
                href = a.get("href")
                if "Download PDF" in a.text:
                    pdf_link = href
                elif "Code" in a.text:
                    code_url = href

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            "code_url": code_url,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [27]:
url = 'https://proceedings.mlr.press/v97/'
DB_PATH = "con_db/ICML_conference_2019.db"
conference_name = 'ICML 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [21]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICML_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [25]:
df_papers = get_www_papers('html/ICML_2019_accepted_papers.html', conference_name)

In [28]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,AReS and MaRS Adversarial and MMD-Minimizing R...,"Gabriele Abbati, Philippe Wenk, Michael A. Osb...",http://proceedings.mlr.press/v97/abbati19a/abb...,https://github.com/gabb7/AReS-MaRS,ICML 2019
1,Dynamic Weights in Multi-Objective Deep Reinfo...,"Axel Abels, Diederik Roijers, Tom Lenaerts, An...",http://proceedings.mlr.press/v97/abels19a/abel...,https://github.com/axelabels/DynMORL,ICML 2019
2,MixHop: Higher-Order Graph Convolutional Archi...,"Sami Abu-El-Haija, Bryan Perozzi, Amol Kapoor,...",http://proceedings.mlr.press/v97/abu-el-haija1...,https://github.com/samihaija/mixhop,ICML 2019
3,Communication-Constrained Inference and the Ro...,"Jayadev Acharya, Clement Canonne, Himanshu Tyagi",http://proceedings.mlr.press/v97/acharya19a/ac...,None,ICML 2019
4,Distributed Learning with Sublinear Communication,"Jayadev Acharya, Chris De Sa, Dylan Foster, Ka...",http://proceedings.mlr.press/v97/acharya19b/ac...,None,ICML 2019


In [29]:
save_to_database(df_papers, conference_name, DB_PATH)

773개의 논문이 ICML 2019에 저장되었습니다.


# ICML 2018

In [34]:
url = 'https://proceedings.mlr.press/v80/'
DB_PATH = "con_db/ICML_conference_2018.db"
conference_name = 'ICML 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [35]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICML_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [36]:
df_papers = get_www_papers('html/ICML_2018_accepted_papers.html', conference_name)

In [37]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Improved Regret Bounds for Thompson Sampling i...,"Marc Abeille, Alessandro Lazaric",http://proceedings.mlr.press/v80/abeille18a/ab...,None,ICML 2018
1,State Abstractions for Lifelong Reinforcement ...,"David Abel, Dilip Arumugam, Lucas Lehnert, Mic...",http://proceedings.mlr.press/v80/abel18a/abel1...,None,ICML 2018
2,Policy and Value Transfer in Lifelong Reinforc...,"David Abel, Yuu Jinnai, Sophie Yue Guo, George...",http://proceedings.mlr.press/v80/abel18b/abel1...,None,ICML 2018
3,INSPECTRE: Privately Estimating the Unseen,"Jayadev Acharya, Gautam Kamath, Ziteng Sun, Hu...",http://proceedings.mlr.press/v80/acharya18a/ac...,None,ICML 2018
4,Learning Representations and Generative Models...,"Panos Achlioptas, Olga Diamanti, Ioannis Mitli...",http://proceedings.mlr.press/v80/achlioptas18a...,None,ICML 2018


In [38]:
save_to_database(df_papers, conference_name, DB_PATH)

621개의 논문이 ICML 2018에 저장되었습니다.


# ICML 2017

In [39]:
url = 'https://proceedings.mlr.press/v70/'
DB_PATH = "con_db/ICML_conference_2017.db"
conference_name = 'ICML 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [40]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICML_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [41]:
df_papers = get_www_papers('html/ICML_2017_accepted_papers.html', conference_name)

In [42]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Uncovering Causality from Multivariate Hawkes ...,"Massil Achab, Emmanuel Bacry, Stéphane Gaı̈ffa...",http://proceedings.mlr.press/v70/achab17a/acha...,None,ICML 2017
1,A Unified Maximum Likelihood Approach for Esti...,"Jayadev Acharya, Hirakendu Das, Alon Orlitsky,...",http://proceedings.mlr.press/v70/acharya17a/ac...,None,ICML 2017
2,Constrained Policy Optimization,"Joshua Achiam, David Held, Aviv Tamar, Pieter ...",http://proceedings.mlr.press/v70/achiam17a/ach...,None,ICML 2017
3,The Price of Differential Privacy for Online L...,"Naman Agarwal, Karan Singh",http://proceedings.mlr.press/v70/agarwal17a/ag...,None,ICML 2017
4,Local Bayesian Optimization of Motor Skills,"Riad Akrour, Dmitry Sorokin, Jan Peters, Gerha...",http://proceedings.mlr.press/v70/akrour17a/akr...,None,ICML 2017


In [43]:
save_to_database(df_papers, conference_name, DB_PATH)

434개의 논문이 ICML 2017에 저장되었습니다.


# 2016

In [45]:
url = 'https://proceedings.mlr.press/v48/'
DB_PATH = "con_db/ICML_conference_2016.db"
conference_name = 'ICML 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [46]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICML_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [47]:
df_papers = get_www_papers('html/ICML_2016_accepted_papers.html', conference_name)

In [48]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,"No Oops, You Won’t Do It Again: Mechanisms for...","Nihar Shah, Dengyong Zhou",http://proceedings.mlr.press/v48/shaha16.pdf,None,ICML 2016
1,Stochastically Transitive Models for Pairwise ...,"Nihar Shah, Sivaraman Balakrishnan, Aditya Gun...",http://proceedings.mlr.press/v48/shahb16.pdf,None,ICML 2016
2,Uprooting and Rerooting Graphical Models,Adrian Weller,http://proceedings.mlr.press/v48/weller16.pdf,None,ICML 2016
3,A Deep Learning Approach to Unsupervised Ensem...,"Uri Shaham, Xiuyuan Cheng, Omer Dror, Ariel Ja...",http://proceedings.mlr.press/v48/shaham16.pdf,None,ICML 2016
4,Revisiting Semi-Supervised Learning with Graph...,"Zhilin Yang, William Cohen, Ruslan Salakhudinov",http://proceedings.mlr.press/v48/yanga16.pdf,None,ICML 2016


In [49]:
save_to_database(df_papers, conference_name, DB_PATH)

322개의 논문이 ICML 2016에 저장되었습니다.


# 2015

In [50]:
url = 'https://proceedings.mlr.press/v37/'
DB_PATH = "con_db/ICML_conference_2015.db"
conference_name = 'ICML 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [51]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICML_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [52]:
df_papers = get_www_papers('html/ICML_2015_accepted_papers.html', conference_name)

In [53]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Stochastic Optimization with Importance Sampli...,"Peilin Zhao, Tong Zhang",http://proceedings.mlr.press/v37/zhaoa15.pdf,None,ICML 2015
1,Approval Voting and Incentives in Crowdsourcing,"Nihar Shah, Dengyong Zhou, Yuval Peres",http://proceedings.mlr.press/v37/shaha15.pdf,None,ICML 2015
2,A low variance consistent test of relative dep...,"Wacha Bounliphone, Arthur Gretton, Arthur Tene...",http://proceedings.mlr.press/v37/bounliphone15...,None,ICML 2015
3,An Aligned Subtree Kernel for Weighted Graphs,"Lu Bai, Luca Rossi, Zhihong Zhang, Edwin Hancock",http://proceedings.mlr.press/v37/bai15.pdf,None,ICML 2015
4,Spectral Clustering via the Power Method - Pro...,"Christos Boutsidis, Prabhanjan Kambadur, Alex ...",http://proceedings.mlr.press/v37/boutsidis15.pdf,None,ICML 2015


In [54]:
save_to_database(df_papers, conference_name, DB_PATH)

270개의 논문이 ICML 2015에 저장되었습니다.


# 2014

In [55]:
url = 'https://proceedings.mlr.press/v32/'
DB_PATH = "con_db/ICML_conference_2014.db"
conference_name = 'ICML 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [56]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ICML_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [57]:
df_papers = get_www_papers('html/ICML_2014_accepted_papers.html', conference_name)

In [58]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,A Discriminative Latent Variable Model for Onl...,"Rajhans Samdani, Kai-Wei Chang, Dan Roth",http://proceedings.mlr.press/v32/samdani14.pdf,None,ICML 2014
1,Kernel Mean Estimation and Stein Effect,"Krikamol Muandet, Kenji Fukumizu, Bharath Srip...",http://proceedings.mlr.press/v32/muandet14.pdf,None,ICML 2014
2,Demystifying Information-Theoretic Clustering,"Greg Ver Steeg, Aram Galstyan, Fei Sha, Simon ...",http://proceedings.mlr.press/v32/steeg14.pdf,None,ICML 2014
3,Covering Number for Efficient Heuristic-based ...,"Zongzhang Zhang, David Hsu, Wee Sun Lee",http://proceedings.mlr.press/v32/zhanga14.pdf,None,ICML 2014
4,The Coherent Loss Function for Classification,"Wenzhuo Yang, Melvyn Sim, Huan Xu",http://proceedings.mlr.press/v32/yanga14.pdf,None,ICML 2014


In [59]:
save_to_database(df_papers, conference_name, DB_PATH)

310개의 논문이 ICML 2014에 저장되었습니다.
